In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler



np.random.seed(1234)

# --------------------------------------------------
# Custom transformers
# --------------------------------------------------

class SelectiveStandardScaler(BaseEstimator, TransformerMixin):
    """
    Apply StandardScaler to selected columns only.
    All other columns are passed through unchanged.
    """
    def __init__(self, cols):
        self.cols = cols
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.cols])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = self.scaler.transform(X[self.cols])
        return X


def aggregate_by_observation(df):
    """
    Aggregate rows by `obs`:
    - Validate categorical columns are constant per obs
    - Average numeric columns
    """
    cat_cols = ["cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]

    # Ensure categorical consistency per observation
    inconsistent = (
        df.groupby("obs")[cat_cols]
          .nunique()
          .ne(1)
          .any()
    )
    if inconsistent.any():
        bad_cols = inconsistent[inconsistent].index.tolist()
        raise ValueError(f"Inconsistent categorical values per obs: {bad_cols}")

    numeric_cols = df.select_dtypes(include="number").columns
    agg = (
        df.groupby("obs", sort=False)[numeric_cols]
          .mean()
          .drop(columns="obs")
    )
    return agg


# --------------------------------------------------
# Preprocessing pipeline
# --------------------------------------------------

NUMERIC_SCALE_COLS = [
    "num_0", "num_1", "num_2",
    "t_0", "t_1", "t_2", "t_3", "t_4"
]

pipeline = Pipeline([
    (
        "coerce_numeric",
        FunctionTransformer(
            lambda df: df.apply(pd.to_numeric, errors="coerce"),
            validate=False
        )
    ),
    (
        "aggregate_obs",
        FunctionTransformer(aggregate_by_observation, validate=False)
    ),
    (
        "scale_selected",
        SelectiveStandardScaler(NUMERIC_SCALE_COLS)
    )
])


# --------------------------------------------------
# Run preprocessing
# --------------------------------------------------

df = pd.read_csv("Project_Data_export/train_competition_2026.csv")
df.sort_values("time", inplace=True)

# Preserve first timestamp per observation
time_by_obs = df.groupby("obs")["time"].first()

processed = pipeline.fit_transform(df)

# Reattach time and sort
processed["time"] = processed.index.map(time_by_obs)
processed.sort_values("time", inplace=True)

# Ensure categorical + id columns are ints
INT_COLS = ["sub_id", "cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]
processed[INT_COLS] = processed[INT_COLS].astype(int)

processed.head()
